## 1. 설치

In [1]:
# %pip install -q langchain langchain-openai pillow pandas python-dotenv

# %pip install -q \
# langchain==0.2.14 \
# langchain-openai==0.1.23 \
# openai>=1.30.0 \
# pillow pandas python-dotenv

## 2. API Key 설정


In [2]:
import os
# os.environ["OPENAI_API_KEY"] = "여기에_API키"
from dotenv import load_dotenv
load_dotenv()

True

## 3. 기본 import

In [3]:
import base64
import json
from pathlib import Path

import pandas as pd
from PIL import Image

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
# from langchain.schema import HumanMessage -- 이건 올드 버전 윗줄이 새버전

## 4. 모델 생성 

In [4]:
llm = ChatOpenAI(
    model="gpt-5-mini",   # 또는 gpt-5.4
     temperature=1
)


# llm = ChatOpenAI(
#     model="gpt-4o-mini",
#     temperature=1
# )

## 5. 이미지 > Base64 변환

In [5]:
def image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

## 6. JSON 파싱함수

In [6]:
def extract_json(text: str):
    text = text.strip()

    try:
        return json.loads(text)
    except:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1:
        return json.loads(text[start:end+1])

    raise ValueError("JSON 파싱 실패")

## 7. LangChain  이미지 분석 함수

In [7]:
def analyze_image_langchain(image_path: str, categories: list[str]):

    base64_img = image_to_base64(image_path)

    category_text = "\n".join([f"- {c}" for c in categories])

    prompt = f"""
당신은 TV 화면 불량 분석 AI입니다.

이미지를 보고:

1. 특징을 설명
2. 카테고리 하나 선택

반드시 JSON만 출력

카테고리:
{category_text}

출력 형식:
{{
  "matched_category": "",
  "confidence": 0.0,
  "summary_ko": "",
  "features": {{
    "horizontal_lines": false,
    "vertical_lines": false,
    "bright_spots": false,
    "dark_spots": false,
    "stain_or_smear": false,
    "brightness_nonuniformity": false
  }},
  "observations": [],
  "reason": ""
}}
"""

    message = HumanMessage(
        content=[
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_img}"
                }
            }
        ]
    )

    response = llm.invoke([message])

    return extract_json(response.content)

## 8. 카테고리 정의

In [8]:
CATEGORIES = [
    "정상",
    "가로줄",
    "세로줄",
    "점불량",
    "얼룩/번짐",
    "밝기불균일",
    "복합불량"
]

## 9. 실행

In [9]:
result = analyze_image_langchain('sample_tv.jpg', CATEGORIES)
result

{'matched_category': '얼룩/번짐',
 'confidence': 0.92,
 'summary_ko': '화면 하단 중앙에 반원 형태의 청색-회색 계열 얼룩이 관찰됨. 주변은 전반적으로 밝은 흰색을 유지하나 해당 부위만 어둡고 색이 번져 보임.',
 'features': {'horizontal_lines': False,
  'vertical_lines': False,
  'bright_spots': False,
  'dark_spots': True,
  'stain_or_smear': True,
  'brightness_nonuniformity': True},
 'observations': ['위치: 화면 하단 중앙에서 상부로 반원 형태로 확장된 영역',
  '형태: 경계가 뚜렷하지 않고 주변으로 퍼지는 번짐(둥근 반원 모양)',
  '색상/톤: 청색-회색 계열로 주변 밝은 흰색보다 현저히 어두움',
  '크기: 화면 전체 높이의 하단 약 15–25% 범위에 해당하는 비교적 큰 영역',
  '다른 결함 없음: 수평·수직 선이나 점형 불량은 관찰되지 않음'],
 'reason': '결함이 화면 특정 부위에 국한되어 경계가 번지듯 퍼져 있고 색상과 밝기가 주변과 달라 점·선 불량이 아닌 얼룩/번짐 유형으로 판단됨. 밝기 불균일도 동반되어 있으나 주요 원인은 국소 얼룩임.'}

## 10. 보기 좋게 출력

In [10]:
def pretty(result):
    print("카테고리:", result["matched_category"])
    print("신뢰도:", result["confidence"])
    print("요약:", result["summary_ko"])
    print()

    print("특징:")
    for k, v in result["features"].items():
        print(f"- {k}: {v}")

    print("\n관찰:")
    for o in result["observations"]:
        print("-", o)

    print("\n이유:", result["reason"])

In [11]:
pretty(result)

카테고리: 얼룩/번짐
신뢰도: 0.92
요약: 화면 하단 중앙에 반원 형태의 청색-회색 계열 얼룩이 관찰됨. 주변은 전반적으로 밝은 흰색을 유지하나 해당 부위만 어둡고 색이 번져 보임.

특징:
- horizontal_lines: False
- vertical_lines: False
- bright_spots: False
- dark_spots: True
- stain_or_smear: True
- brightness_nonuniformity: True

관찰:
- 위치: 화면 하단 중앙에서 상부로 반원 형태로 확장된 영역
- 형태: 경계가 뚜렷하지 않고 주변으로 퍼지는 번짐(둥근 반원 모양)
- 색상/톤: 청색-회색 계열로 주변 밝은 흰색보다 현저히 어두움
- 크기: 화면 전체 높이의 하단 약 15–25% 범위에 해당하는 비교적 큰 영역
- 다른 결함 없음: 수평·수직 선이나 점형 불량은 관찰되지 않음

이유: 결함이 화면 특정 부위에 국한되어 경계가 번지듯 퍼져 있고 색상과 밝기가 주변과 달라 점·선 불량이 아닌 얼룩/번짐 유형으로 판단됨. 밝기 불균일도 동반되어 있으나 주요 원인은 국소 얼룩임.


## 폴더안 내용 확인

In [12]:
from pathlib import Path

list(Path("images").glob("*"))

[PosixPath('images/sample_tv.jpg'),
 PosixPath('images/sample_tv3.jpg'),
 PosixPath('images/sample_tv2.jpg'),
 PosixPath('images/sample_tv4.png')]

## 11. 배치 함수 수정본

In [13]:
from pathlib import Path
import pandas as pd

def batch(folder="images"):

    rows = []
    folder_path = Path(folder)

    if not folder_path.exists():
        raise ValueError(f"{folder} 폴더가 없습니다")

    files = list(folder_path.glob("*"))

    if len(files) == 0:
        print("이미지 파일이 없습니다")
        return pd.DataFrame()

    for path in files:
        if path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        try:
            r = analyze_image_langchain(str(path), CATEGORIES)

            rows.append({
                "file": path.name,
                "category": r["matched_category"],
                "confidence": r["confidence"],
                "summary": r["summary_ko"]
            })

        except Exception as e:
            rows.append({
                "file": path.name,
                "category": "ERROR",
                "confidence": None,
                "summary": str(e)
            })

    return pd.DataFrame(rows)

## 12. 배치 실행

In [14]:
df = batch("images")
df

,file,category,confidence,summary
0,sample_tv.jpg,얼룩/번짐,0.91,화면 하단 중앙에서 반원형의 어두운 얼룩(푸른빛)이 관찰되며 주변보다 밝기가 낮고 ...
1,sample_tv3.jpg,밝기불균일,0.90,전체가 파란색 화면이지만 중앙에 넓은 타원형의 밝은 영역이 존재하고 가장자리로 갈수...
2,sample_tv2.jpg,점불량,0.95,화면 상단 좌측 부근에 작은 검은 점(픽셀 또는 픽셀 군)이 단발적으로 관찰됨. 그...
3,sample_tv4.png,정상,0.94,"화면 전체가 균일한 단색(짙은 파란색)으로 보이며, 눈에 띄는 가로/세로 줄, 점 ..."


## 13. 결과 저장

In [15]:
df.to_csv("output/result.csv", index=False, encoding="utf-8-sig")